In [1]:
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

In [2]:
N = 1000
C,H_in,W_in = 1,28,28
m = 10
rng = np.random.default_rng(0)

In [3]:
from tensorflow.keras.datasets import mnist

2026-03-04 14:05:45.404631: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [4]:
def one_hot(y,num_classes=10):
    Y = np.zeros((num_classes,y.shape[0]),
                 dtype=np.float32)
    Y[y,np.arange(y.shape[0])] = 1.0
    return Y

In [5]:
(Xtr,Ytr), (Xte,Yte) = mnist.load_data()

Xtr = (Xtr.astype(np.float32)/255.0)[:,None,:,:] #(N,1,28,28)
Xte = (Xte.astype(np.float32)/255.0)[:,None,:,:]

idx = rng.permutation(Xtr.shape[0])
n_val = int(1e4)
val_idx = idx[:n_val]
train_idx = idx[n_val:]


X_train, y_train = Xtr[train_idx], Ytr[train_idx]
X_val,y_val = Xtr[val_idx], Ytr[val_idx]

Y_train = one_hot(y_train,10)
Y_val = one_hot(y_val,10)

In [6]:
#FORWARD

def relu_forward(Z):
    return np.maximum(0,Z)

def conv2_forward(X,W,b,pad=0,stride=1):
    if pad > 0: #(N,C,H+2p,W+2p)
        X_pad = np.pad(X, ((0,0),(0,0),(pad,pad),(pad,pad)), 
                       mode="constant")
    else:
        X_pad = X
                                           #(Kh,Kw)
    windows = sliding_window_view(X_pad,(W.shape[2],W.shape[3]),axis=(2,3))

    if stride > 1:
        windows = windows[:,:,::stride,::stride,:,:]
    
    Z = np.einsum("nchwkl,fckl->nfhw",windows,W,optimize=True)
    Z = Z + b.reshape(1, -1, 1, 1) #bias
    return Z, X_pad

def dense_forward(X,W,b):
    # X:(d_in,N) -> (d_out,N)
    return W@X+b

def gap_forward(X):
    #X:(N,C,H,W) -> (C,N)
    G = np.mean(X,axis=(2,3)).T #(N,C)->(C,N)
    cache = X.shape
    return G,cache

def bn_forward(X,gamma,beta,r_mean,r_var,mode):
    #X = (N,C,H,W) y,b = C
    eps = 1e-12
    a = 0.3

    g = gamma.reshape(1,-1,1,1)
    b = beta.reshape(1,-1,1,1)

    if mode == "train":
        mu = np.mean(X, axis=(0,2,3),keepdims=True)
        var = np.mean((X-mu)**2,axis=(0,2,3),keepdims=True)

        
        X_hat = (X-mu)/(np.sqrt(var+eps))
        Y = g * X_hat + b
        r_mean = (1-a)*r_mean+a*mu.reshape(-1)
        r_var = (1-a)*r_var+a*var.reshape(-1)

        cache = (X,X_hat,mu,var,gamma,eps)
        return Y, r_mean,r_var,cache
    else:
        mu = r_mean.reshape(1,-1,1,1)
        var = r_var.reshape(1,-1,1,1)

        X_hat = (X - mu) / np.sqrt(var + eps)   
        Y = g * X_hat + b

        cache = (X,X_hat,mu,var,gamma,eps)
        return Y, r_mean,r_var,cache
                                              
def resblock_forward(X,layer,p,mode):

    pad = layer.get("pad", 0)
    stride = layer.get("stride", 1)
     # main: conv -> bn -> relu -> conv -> bn

    z1, xpad1 = conv2_forward(X, p["W1"], p["b1"], pad=pad, stride=stride)
    u1, rm1, rv1, bn1_cache = bn_forward(z1, p["bn1"]["gamma"], p["bn1"]["beta"],
        p["bn1"]["running_mean"], p["bn1"]["running_var"], mode)
    p["bn1"]["running_mean"], p["bn1"]["running_var"] = rm1, rv1

    a1_in = u1
    a1 = relu_forward(a1_in)

    z2, xpad2 = conv2_forward(a1, p["W2"], p["b2"], pad=pad, stride=1)

    m, rm2, rv2, bn2_cache = bn_forward(
        z2, p["bn2"]["gamma"], p["bn2"]["beta"],
        p["bn2"]["running_mean"], p["bn2"]["running_var"], mode
    )
    p["bn2"]["running_mean"], p["bn2"]["running_var"] = rm2, rv2

    if layer.get("projection", False):
        s, xpad_skip = conv2_forward(X, p["Wskip"], p["bskip"], pad=0, stride=stride)
        skip_cache = ("proj", xpad_skip)
    else:
        s = X
        skip_cache = ("id", None)

    t = m + s
    y_in = t
    y = relu_forward(y_in)

    cache = {
        "c1": xpad1,
        "bn1": bn1_cache,
        "r1": a1_in,
        "c2": xpad2,
        "bn2": bn2_cache,
        "skip": skip_cache,
        "rout": y_in
    }
    return y, cache

In [7]:
#BACKWARD
def relu_backward(dA,Z):
    return dA*(Z>0)

def conv2d_backward(dZ,#(N,F,Ho,Wo)
                    X_pad,#(N,C,Hp,Wp)
                    W,#(F,C,Kh,Kw)
                    pad=0,stride=1):
    N,F,H_out,W_out = dZ.shape
    Fw,C,Kh,Kw = W.shape
    assert Fw==F

    db = np.sum(dZ,axis=(0,2,3)).reshape(F,1)

    #stride = 1
    H_full = X_pad.shape[2] - Kh + 1
    W_full = X_pad.shape[3] - Kw + 1

    if stride>1:
        dZ_dill = np.zeros((N,F,H_full,W_full))
        dZ_dill[:,:,::stride,::stride] = dZ
    else:
        dZ_dill = dZ

    X_windows = sliding_window_view(X_pad,(Kh,Kw),
                                           axis=(2,3))
    #X_windows: (N, C, H_full, W_full, Kh, Kw)
    dW = np.einsum("nfhw,nchwkl->fckl", dZ_dill, X_windows)

    dZ_pad = np.pad(dZ_dill,((0,0),(0,0),(Kh-1,Kh-1),(Kw-1,Kw-1)),
                    mode="constant")
    dZ_windows = sliding_window_view(dZ_pad,(Kh,Kw),axis=(2,3))
    #dZ_windows: (N, F, H_p, W_p, Kh, Kw)
    W_rot = W[:,:,::-1,::-1] #180 filters rot

    dX_pad = np.einsum("nfhwkl,fckl->nchw", dZ_windows, W_rot)
    dX = dX_pad[:, :, pad:-pad, pad:-pad] if pad > 0 else dX_pad
    #(N,C,H,W)
    return dX, dW, db

def bn_backward(dY,cache):
    X,X_hat,mu,var,gamma,eps = cache
    N,C,H,W = dY.shape
    m=N*H*W

    g = gamma.reshape(1,-1,1,1)
    if mu.ndim == 1:
        mu=mu.reshape(1,-1,1,1)
    if var.ndim == 1:
        var = var.reshape(1,-1,1,1)

    #db,dy
    dbeta = np.sum(dY,axis=(0,2,3)) #(C,)
    dgamma = np.sum(dY*X_hat,axis=(0,2,3)) #(C,)

    dX_hat = dY*g
    inv_std = 1.0/np.sqrt(var+eps)

    s1 = np.sum(dX_hat,axis=(0,2,3),keepdims=True) #(1,C,1,1)
    s2 = np.sum(dX_hat*X_hat,axis=(0,2,3),keepdims=True) #(1,C,1,1)

    dX = (1.0/m)*inv_std*(m*dX_hat-s1-X_hat*s2) #(N,C,H,W)
    return dX,dgamma,dbeta

def dense_backward(dZ,X,W):
    dX = W.T@dZ
    dW = dZ@X.T
    db=np.sum(dZ,axis=1,keepdims=True)
    return dX,dW,db

def gap_backward(dG,cache):
    N,C,H,W = cache
    dX = (dG.T[:,:,None,None]/(H*W))*np.ones((N,C,H,W)) #(C,N)->(N,C,1,1)->(N,C,H,W)
    return dX

def resblock_backward(dY,layer,p,cache):
    pad = layer.get("pad", 0)
    stride = layer.get("stride", 1)
    grads = {}

    #1) final ReLU
    dT = relu_backward(dY,cache["rout"])

    #2) add backward
    dM = dT
    dS = dT

    #skip branch
    tag, xpad_skip=cache["skip"]
    if tag == "proj":
        dX_skip,dWskip,dbskip = conv2d_backward(dS, xpad_skip, 
                p["Wskip"], pad=0, stride=layer["stride"])
        grads["Wskip"] = dWskip
        grads["bskip"] = dbskip
    else:
        dX_skip = dS
    
    #main branch
    #bn2
    dZ2,dgamma2,dbeta2 = bn_backward(dM,cache["bn2"])
    grads["gamma2"] = dgamma2
    grads["beta2"] = dbeta2

    #conv2
    dA1,dW2,db2 = conv2d_backward(dZ2,cache["c2"],
        p["W2"],pad=layer["pad"],stride=1)
    grads["W2"] = dW2
    grads["b2"] = db2

    #relu1
    dU1 = relu_backward(dA1,cache["r1"])

    #bn1
    dZ1, dgamma1, dbeta1 = bn_backward(dU1, cache["bn1"])
    grads["gamma1"] = dgamma1
    grads["beta1"]  = dbeta1

    # conv1 (stride=layer["stride"], pad=layer["pad"])
    dX_main, dW1, db1 = conv2d_backward(
        dZ1, cache["c1"], p["W1"],  pad=layer["pad"], stride=layer["stride"]
    )
    grads["W1"] = dW1
    grads["b1"] = db1
    
    #5)sum grads
    dX = dX_main+dX_skip

    return dX,grads


In [8]:
def init_network(arch,input_shape,rng):
    params = [None]*len(arch)
    current_shape = input_shape

    for l, layer in enumerate(arch):
        l_type = layer["type"]

        if l_type=="conv":
            C_in,H,W = current_shape
            F,K = layer["filters"],layer["kernel"]
            pad,stride=layer.get("pad",0),layer.get("stride",1)

            std=np.sqrt(2.0/(C_in*K*K))
            params[l]={
                "W":rng.normal(0.0,std,size=(F,C_in,K,K)),
                "b":np.zeros((F,1))
            }
            current_shape = (F,(H+2*pad-K)//stride+1,
                             (W+2*pad-K)//stride+1)
            
        elif l_type == "bn":
            C = layer.get("filters",current_shape[0])
            params[l] = {
                "gamma": np.ones((C,)),
                "beta": np.zeros((C,)),
                "running_mean":np.zeros(C,),
                "running_var": np.ones((C,))
            }

        elif l_type == "relu":
            params[l] = None
        
        elif l_type == "gap":
            #(C,H,W) -> (C,)
            C,H,W = current_shape
            params[l] = None
            current_shape = (C,)
            
        elif l_type == "dense":
            d_in, d_out = current_shape[0], layer["units"]
            std = np.sqrt(2.0 / d_in)
            params[l] = {
                "W": rng.normal(0.0, std, size=(d_out, d_in)),
                "b": np.zeros((d_out, 1))
            }
            current_shape = (d_out,)

        elif l_type == "resblock":
            C_in,H,W = current_shape
            F,K = layer["filters"],layer["kernel"]
            pad,stride = layer.get("pad",0), layer.get("stride",1)
            projection = layer.get("projection",False)

            #conv1 Cin->F
            std1=np.sqrt(2.0/(C_in*K*K))
            W1=rng.normal(0.0,std1,size=(F,C_in,K,K))
            b1=np.zeros((F,1))

            #bn1 params
            bn1={
                "gamma": np.ones((F,)),
                "beta": np.zeros((F,)),
                "running_mean": np.zeros((F,)),
                "running_var": np.ones((F,))
            }

            #conv2 F->F
            std2 = np.sqrt(2.0 / (F * K * K))
            W2 = rng.normal(0.0, std2, size=(F, F, K, K))
            b2 = np.zeros((F, 1))

            # bn2 params
            bn2 = {
                "gamma": np.ones((F,)),
                "beta": np.zeros((F,)),
                "running_mean": np.zeros((F,)),
                "running_var": np.ones((F,))
            }

            p = {"W1":W1, "b1":b1, "bn1":bn1,
                 "W2":W2, "b2":b2, "bn2":bn2}
            
            if projection:
                stds = np.sqrt(2.0 / (C_in * 1 * 1))
                p["Wskip"] = rng.normal(0.0, stds,size=(F,C_in,1,1))
                p["bskip"] = np.zeros((F, 1))

            params[l] = p

            H_out = (H + 2 * pad - K) // stride + 1
            W_out = (W + 2 * pad - K) // stride + 1
            current_shape = (F, H_out, W_out)

    return params

In [9]:
def forward_net(X, arch, params,mode):
    cache =[]
    A = X
    
    for l, layer in enumerate(arch):
        l_type = layer["type"]

        if l_type == "conv":
            Z, X_pad = conv2_forward(A, params[l]["W"], params[l]["b"], 
                                      layer.get("pad",0), layer.get("stride",1))
            cache.append(("conv", X_pad))
            A = Z

        elif l_type == "relu":
            cache.append(("relu",A))
            A = relu_forward(A)

        elif l_type == "bn":
            Y,r_mean,r_var,bn_cache = bn_forward(A,params[l]["gamma"],
                                                 params[l]["beta"],
                                                 params[l]["running_mean"],
                                                 params[l]["running_var"],
                                                 mode)
            params[l]["running_mean"] = r_mean
            params[l]["running_var"] = r_var
            cache.append(("bn",bn_cache))
            A = Y

        elif l_type == "dense":
            cache.append(("dense",A))
            A = dense_forward(A,params[l]["W"],params[l]["b"])

        elif l_type == "resblock":
            Y,b_cache = resblock_forward(A,layer,params[l],mode)
            cache.append(("resblock",b_cache))
            A=Y
            
        elif l_type =="gap":
            G,gap_cache = gap_forward(A)    
            cache.append(("gap",gap_cache))
            A=G
            
    return A, cache

In [10]:
def backward_net(dA, cache, arch, params):
    grads = [None] * len(arch)
    
    for l in reversed(range(len(arch))):
        l_type, layer = arch[l]["type"], arch[l]
        
        tag,c = cache[l]

        if l_type == "dense":
            A = c #("dense",A_in)
            dA, dW, db = dense_backward(dA, A, params[l]["W"])
            grads[l] = {"W": dW, "b": db}

        elif l_type == "gap":
            #("gap", (N,C,H,W))
            dA = gap_backward(dA,c)

        elif l_type == "resblock":
            #("resblokc",bloack_cache)
            dA,gblock = resblock_backward(dA,layer,params[l],c)
            grads[l] = gblock
        
        elif l_type == "relu":
            #("relu",X_in)
            dA = relu_backward(dA, c)

        elif l_type == "bn":
            #("bn",bn_cache)
            dA,dgamma,dbeta = bn_backward(dA,c)
            grads[l] = {"gamma":dgamma,"beta":dbeta}
        
        elif l_type == "conv":
            #("conv",X_pad)
            X_pad = c    
            dA, dW, db = conv2d_backward(dA, X_pad, params[l]["W"], 
                                         layer.get("pad",0), layer.get("stride",1))
            grads[l] = {"W": dW, "b": db}
            
    return grads

def update_net(params, grads, lr):
        for p, g in zip(params, grads):
            if g is None:
                continue
        
            #conv/dense
            if "W" in g:
                p["W"] -= lr * g["W"]
                p["b"] -= lr * g["b"]

            #BN(separat)
            if "gamma" in g:
                p["gamma"] -= lr*g["gamma"]
                p["beta"] -= lr*g["beta"]

            #resblock
            if "W1" in g:
                p["W1"]-=lr*g["W1"]
                p["b1"]-=lr*g["b1"]
                p["W2"]-=lr*g["W2"]
                p["b2"]-=lr*g["b2"]

                p["bn1"]["gamma"] -= lr*g["gamma1"]
                p["bn1"]["beta"] -= lr*g["beta1"]
                p["bn2"]["gamma"] -= lr*g["gamma2"]
                p["bn2"]["beta"] -= lr*g["beta2"]

            if "Wskip" in g:
                p["Wskip"]-=lr*g["Wskip"]
                p["bskip"]-=lr*g["bskip"]
        return params

In [11]:
def softmax(Z):
    Zs = Z - np.max(Z,axis=0,keepdims=True)
    expZ = np.exp(Zs)
    return expZ/np.sum(expZ,axis=0,keepdims=True)

def loss_ce_from_logits(logits,Y):
    P = softmax(logits)
    eps = 1e-12
    loss = -np.mean(np.sum(Y*np.log(P+eps),axis=0))
    return loss,P

In [12]:
def fit(X_train, Y_train, arch, params, lr=2e-2, epochs=5, batch_size=64, X_val=None, 
        Y_val=None, y_val_int=None, log_every=20):
    N_samples = X_train.shape[0]
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    num_batches = (N_samples + batch_size - 1) // batch_size

    for ep in range(1, epochs + 1):
        idx = np.random.permutation(N_samples)
        train_losses =[]

        for batch_no, start in enumerate(range(0, N_samples, batch_size), start=1):
            batch_idx = idx[start:start+batch_size]
            Xb, Yb = X_train[batch_idx], Y_train[:, batch_idx]

            #Forward -> Loss -> Backward -> Update
            #forward
            logits, cache = forward_net(Xb, arch, params,mode="train")
            #Loss+softmax
            L,P = loss_ce_from_logits(logits, Yb)
            train_losses.append(float(L))
            
            dA0 = (P-Yb)/Xb.shape[0]
            #backward + upd
            grads = backward_net(dA0, cache, arch, params)
            params = update_net(params, grads, lr)

            if log_every and (batch_no % log_every == 0 or batch_no == num_batches):
                seen = min(start + batch_size, N_samples)
                avg_loss = float(np.mean(train_losses))
                print(f"Epoch {ep}/{epochs} | Batch {batch_no}/{num_batches} | Seen {seen}/{N_samples} | Train CE: {avg_loss:.4f}", flush=True)
            

        train_loss = float(np.mean(train_losses))
        history["train_loss"].append(train_loss)

        msg = f"Epoch {ep}/{epochs} | Train CE: {train_loss:.4f}"

        #val
        if X_val is not None and Y_val is not None:
            logits_val, _ = forward_net(X_val, arch, params,mode="eval")
            val_loss,val_P = loss_ce_from_logits(logits_val,Y_val)
            history["val_loss"].append(float(val_loss))

            if y_val_int is not None:
                pred = np.argmax(val_P,axis=0)
                acc = float(np.mean(pred == y_val_int))
                history["val_acc"].append(acc)
                msg += f" | Val CE: {val_loss:.4f} | Val Acc: {acc:.4f}"
            else:
                msg += f" | Val CE: {val_loss:.4f}"

        print(msg, flush=True)
            
    return params, history

In [13]:
arch =[
    # Input: 1 x 28 x 28
    {"type": "conv", "filters": 4, "kernel": 3, "pad": 1, "stride": 1}, # -> 4x28x28
    {"type": "bn", "filters":4},
    {"type": "relu"},
    
    {"type": "resblock", "filters": 4,  "kernel": 3, "pad": 1, "stride": 1, "projection": False},
    {"type": "resblock", "filters": 4,  "kernel": 3, "pad": 1, "stride": 1, "projection": False},
    
    # -> 8 x 14 x 14
    {"type": "resblock", "filters": 8,  "kernel": 3, "pad": 1, "stride": 2, "projection": True},  
    {"type": "resblock", "filters": 8,  "kernel": 3, "pad": 1, "stride": 1, "projection": False},

    # -> 16 x 7 x 7
    {"type": "resblock", "filters": 16, "kernel": 3, "pad": 1, "stride": 2, "projection": True},  
    
    {"type": "resblock", "filters": 16, "kernel": 3, "pad": 1, "stride": 1, "projection": False},
    
    {"type": "gap"},# 16 x 7 x 7 -> 16
    {"type": "dense", "units": m},# 16 -> m                       
]

In [14]:
params = init_network(arch, input_shape=(C,H_in,W_in), rng=rng)
params, history = fit(X_train, Y_train, arch, params,
                      lr=2e-2, epochs=5, batch_size=64,
                      X_val=X_val, Y_val=Y_val, y_val_int=y_val)

Epoch 1/5 | Batch 20/782 | Seen 1280/50000 | Train CE: 2.7203
Epoch 1/5 | Batch 40/782 | Seen 2560/50000 | Train CE: 2.5165
Epoch 1/5 | Batch 60/782 | Seen 3840/50000 | Train CE: 2.4083
Epoch 1/5 | Batch 80/782 | Seen 5120/50000 | Train CE: 2.3306
Epoch 1/5 | Batch 100/782 | Seen 6400/50000 | Train CE: 2.2718
Epoch 1/5 | Batch 120/782 | Seen 7680/50000 | Train CE: 2.2283


KeyboardInterrupt: 